# 8.4 Vector 性能（上）：VF 优化

[第7章 7.6](../07_performance_basics/07.06_inner_pipeline.ipynb) 已经搭好了 RegBase 编程的骨架：`GM → UB → RegisterFile → UB → GM`，vLoop 按 256B 寄存器宽度切分。本节进入**指令级优化**——同样的功能，指令数可以从 N 条压到 N/3 条。

## 1. VF 指令开销的直觉

一条 VF 指令吃 64 个 FP32（256B 寄存器）。假设某 element-wise 操作处理 `4096` 个元素：

```
循环次数 = 4096 / 64 = 64 轮
每轮 = Load + Load + 计算 + Store（至少 4 条）
总计 ≈ 256 条 VF 指令
```

VF 没有分支预测、没有乱序——**源代码里写了几条指令，硬件就按顺序执行几条**。所以 VF 优化的本质就一句话：**少写指令**。

## 2. 五个立竿见影的优化点

### 优化点 1：循环不变量外提

与 C 语言循环优化完全同构——`Duplicate(vregScale, scale)` 这种每轮都重复、结果相同的指令，提一句到 for 外，全循环只剩 1 次。

```cpp
// 优化前（每轮都 Duplicate）
for (i...) {
    RegTensor<float> vScale;
    Duplicate(vScale, scale);      // 循环 64 轮 = 64 条
    ...
}

// 优化后（外提一次）
RegTensor<float> vScale;
Duplicate(vScale, scale);          // 仅 1 条
for (i...) {
    ...
}
```

### 优化点 2：尾块用 MaskReg 兜底，别拆两个循环

数据量不是 64 整数倍时，常见写法是「整块循环 + 尾块单独一段代码」。更好的做法是**全程一个循环，最后一轮换 MaskReg**：

```cpp
MaskReg pregAll = CreateMask<float, MaskPattern::ALL>();
MaskReg pregTail = CreateMask<float, MaskPattern::ALL>();  // 按尾长改位
for (i = 0; i < repeatTimes; i++) {
    MaskReg &preg = (i == repeatTimes - 1 && hasTail) ? pregTail : pregAll;
    // 同一套指令，只换掩码
}
```

### 优化点 3：能合并的 Load/Store 不拆

同一块数据被多条指令分别搬运时，合并为一次 `DataCopy` 进寄存器后，用**不同 RegTensor 视角**消费。

### 优化点 4：广播类搬运用专用 LoadDist

per-token scale 这类「一行一个系数」的场景，用 `DIST_BRC_B32` 广播装载，替代先装载再逐行 Duplicate。

### 优化点 5：计算图重排，削减中间量

例如 `x * (a / b)` 且 a、b 是标量：先在 Scalar 侧算 `a/b`（一条标量除法），再广播乘——**把 Vector 除法变乘法**。除法比乘法贵数倍，这类重排积少成多。

## 3. 以 MulQr_VF 为例做一次完整优化

原实现（第7章 7.6 的版本）每轮：Load 源 → Load scale（广播）→ Duplicate 量化系数 → Div → Mul → Store，共 6 条。逐项检查五个优化点：

| 优化点 | 命中？ | 改法 |
|--|--|--|
| 不变量外提 | **是** | `Duplicate(quantScale)` 每轮相同 → 外提 |
| MaskReg 尾块 | 是 | 已有 |
| 合并搬运 | 视数据布局 | 源与 scale 相邻时可合并 Load |
| 专用广播 | **是** | scale 装载已用 BRC 广播 |
| 计算重排 | **是** | `quantScale / dequantScale` 每轮同值 → 循环外算成标量 |

## 动手实验：优化前后的指令数对比


In [ ]:
# 动手：MulQr_VF 优化前后的指令数建模
import numpy as np

total_elems = 4096        # 待处理元素数（fp32）
REG_ELEMS = 64            # 一条 VF 指令处理的 FP32 数（256B 寄存器）

rounds = -(-total_elems // REG_ELEMS)

# ---- 优化前：每轮 6 条指令 ----
per_round_before = 6      # Load x2 + Duplicate + Div + Mul + Store
total_before = per_round_before * rounds

# ---- 优化后 ----
# 不变量外提: Duplicate 与 Div(quantScale/dequantScale) 均外提为标量计算
hoisted = 2               # Duplicate 1 条 + 标量 Div（Scalar 侧，几乎免费）
per_round_after = 4       # Load x2 + Mul + Store
total_after = hoisted + per_round_after * rounds

print(f"元素数 {total_elems}, 循环轮数 {rounds}")
print(f"优化前指令数: {total_before}")
print(f"优化后指令数: {total_after}")
print(f"削减比例:    {1 - total_after/total_before:.1%}")


### 观察结论

外提两条循环不变量后，指令数降 1/3。对 FA kernel 来说，softmax 的 SoftmaxV2/Max/Sum、update 的 Mul/Adds 都能吃到同样的收益。

## 4. 指令编排：让相邻指令不打架

指令数之外，还有**指令间依赖**这一层：

- 相邻两条指令若操作同一块 UB（写后读），会插等待；
- 好的编排是**交错两组独立数据**（软件双缓冲）：指令序列 `A.load → B.load → A.calc → B.calc → A.store → B.store`；
- 这与核间 PingPong 是同一个思想，只是搬到了寄存器层面。

## 5. VF 定位要点（回顾 + 延伸）

[第7章 7.6](../07_performance_basics/07.06_inner_pipeline.ipynb) 给过 VF 定位三步。这里补一条工程习惯：**VF 代码无法单步调试，每写一段就用「GM 进 → UB 出」的旁路打印验证一遍**，不要攒到最后对拍时才发现 UB 上的数据就错了。

## 本节要点

- VF 优化 = 少写指令：五个优化点（外提、MaskReg 尾块、合并搬运、专用广播、计算重排）；
- 除法变乘法、逐行 Duplicate 变广播装载，这类小改动能稳定拿到两位数百分比收益；
- 指令编排上用寄存器级双缓冲消除相邻依赖；
- VF 没有调试器，写一段验一段。

## 小测验

1. VF 优化的本质为什么是「少写指令」？与 CPU 的乱序执行对比说明。
2. `x * (a/b)` 中 a、b 为标量时，怎么重排最省 VF 指令？
3. 数据量 4100 个 FP32、单指令 64 个，写出 MaskReg 方案的循环结构（伪代码）。
4. 寄存器级双缓冲为什么能消除相邻指令的等待？

>>> 参考答案见 [answer/08.04_answer.txt](answer/08.04_answer.txt)
